# LLM-Assisted Thematic Analysis

Goal: use an LLM **only where it adds measurable value** over the non-LLM methods from Days 2–4.

Days 3–4 found the themes without an LLM, and BERTopic already gives keywords for every topic. What those methods can't do:

| Limitation found earlier | LLM task that addresses it |
|---|---|
| A human had to read every topic and write its name and description (Day 4, 3a) | **Task A**: name topics from keywords and representative comments |
| BERTopic gives **one** topic per comment, but 194 comments (27%) may discuss several ideas (Day 1) | **Task B**: multi-label theme × sentiment classification |
| The dataset has **one** sentiment per comment, even for "teaching is good but paper checking is bad" | **Task B**: a sentiment *per theme* |
| Keywords and counts don't say *what* students want changed | **Task C**: summaries grounded in real comments |

What the LLM is **not** used for: finding the themes (BERTopic did that, reproducibly and for free), counting (pandas does that exactly), or reading the whole dataset at once.

## Rules for every LLM call

These come from the plan, and each one has a reason:
1. **Structured output.** Every call has a pydantic schema, sent as a JSON schema (`output_config.format` for Claude, `--output-schema` for Codex). The model is then forced to return valid JSON in that shape, so there's no fragile text parsing. Theme names in Task B are an `enum` built from the taxonomy, so the model *can't* invent a theme.
2. **Validate anyway.** The schema doesn't cover everything. `confidence` must be between 0 and 1, evidence must really be copied from the comment, and cited comment ids must exist. These checks run in code. A response that fails them is **recorded, not raised**, so one bad answer doesn't stop a batch.
3. **Evidence ids.** Task B quotes its evidence and Task C cites `supporting_comment_ids`, so every claim can be checked against a real comment.
4. **Log every call.** Prompt version, model, effort, latency, tokens, cost and status, including failed responses. If a result looks wrong later, the log says exactly which prompt and model produced it.
5. **Humans review uncertain output.** Low confidence, a new theme, ungrounded evidence or a failed call sends the item to an analyst review queue instead of accepting it.
6. **Comments are untrusted input.** A comment could contain text like "ignore the instructions". Comments go inside `<comments>` tags, and every prompt says to treat them as data.

## Code layout

The LLM code lives in `src/llm/`, not in the notebook, because Day 7's API will reuse it:
- `prompts.py`: every prompt, each with a version (`name_theme_v1`, ...)
- `client.py`: `call_llm`, the one function every task goes through (schema, validation, logging, cost). It has two backends, picked by the model name: `claude-...` calls the Anthropic API, `gpt-...` runs the local Codex CLI
- `name_theme.py`, `classify_theme.py`, `summarise_theme.py`: one task each

`tests/test_llm.py` checks the validation logic with a fake client, without calling the API (`uv run pytest tests/test_llm.py`).

Input: Day 4's `results/` (taxonomy, topics, representative comments, topic assignments). Outputs are written to `results/`.

## 0. Setup

**Two ways to run the LLM.** `MODEL` picks the backend:

| | `gpt-5.6-luna` (default) | `claude-opus-5-5` |
|---|---|---|
| Runs through | the local Codex CLI | the Anthropic API |
| Needs | `codex` installed and logged in with ChatGPT | `ANTHROPIC_API_KEY` in `.env` |
| Cost | your ChatGPT plan's usage limits; `cost_usd` is empty | paid per token; `cost_usd` is exact |
| Per call | ~5–12 s, ~12,000 input tokens (mostly Codex's own agent instructions) | ~2–5 s, ~1,000 input tokens |

Both get the same prompts, the same JSON schema and the same validation, so results are comparable. Run the notebook once per backend to compare them.

Codex is a coding agent, so `client.py` runs it read-only, in an empty temporary folder: it can't see the dataset, the project or `.env`.

**Effort.** Both models think before answering; `effort` controls how much. It's `low` for naming and classifying (short, well-defined tasks) and `medium` for summaries.

**Calls.** About 173: 1 test, 16 naming, 150 classification and 6 summaries. Run section 1's single test call first and check it works before running the rest.

**Claude only:** `client.py` sends `fallbacks: "default"`, so if a safety classifier declines a request (unlikely for student feedback), the API re-runs it on another model. The SDK retries rate limits and server errors itself; `max_retries=5` gives a long batch more chances. For Claude, create `.env` in the project root (it's in `.gitignore`):

```
ANTHROPIC_API_KEY=sk-ant-...
```

In [1]:
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import anthropic
import pandas as pd
import yaml
from dotenv import load_dotenv

sys.path.insert(0, "..")
from src.llm.classify_theme import GENERAL, classify_comment
from src.llm.name_theme import name_theme
from src.llm.summarise_theme import summarise_theme

RESULTS = Path("../results")
MODEL = "gpt-5.6-luna"  # or "claude-opus-5-5" (needs ANTHROPIC_API_KEY)
REVIEW_THRESHOLD = 0.7  # confidence below this goes to analyst review

load_dotenv("../.env")
client = anthropic.Anthropic(max_retries=5)
log = []  # one record per LLM call

## 1. Load Day 4's results

In [2]:
df = pd.read_csv("../data/processed/cleaned_feedback.csv")
taxonomy = yaml.safe_load((RESULTS / "theme_taxonomy_v1.yaml").read_text())
topics = pd.read_csv(RESULTS / "topic_summary.csv", index_col="topic")
examples = pd.read_csv(RESULTS / "representative_comments.csv")
assignments = pd.read_csv(RESULTS / "topic_assignments.csv", index_col="comment_id")

**Test call.** One call to check that the key, model and schema work, and what a single call costs.

In [3]:
comments = examples.query("topic == 0")["text"].tolist()
parsed, record = name_theme(client, topics.loc[0, "keywords"], comments, model=MODEL)
log.append(record)
parsed, record["cost_usd"]

(ThemeName(theme='Teaching delivery and interaction', description='Comments assess the quality of lectures and teaching, teacher punctuality, and interaction between teachers and students.', generic_feedback=False, confidence=0.98),
 nan)

**Result:** The test call worked. It returned a valid `ThemeName` for topic 0, "Teaching delivery and interaction", with confidence 0.98 and `generic_feedback=False`. `cost_usd` is `nan` because Codex runs on the ChatGPT plan, not per token.

**How to read it:** A parsed object (not `None`) means the backend, model and schema all work. If `parsed` were `None`, `record["status"]` and `record["raw_output"]` would say why. The name covers the same subject as Day 4's "Teaching Quality", so the prompt works before you spend 172 more calls.

## 2. Task A: theme naming

Give the LLM what Day 4 gave you, a topic's keywords and its 5 most typical comments, and ask for a theme name, a description and a confidence.

How to measure value: Day 4's `TOPICS` table is a human answer to the same question. The key check is whether the LLM agrees with you about which topics are **general feedback** (a tone, not a theme). Disagreements and low-confidence names go to review.

In [4]:
rows = []
for topic, row in topics.iterrows():
    comments = examples.query("topic == @topic")["text"].tolist()
    parsed, record = name_theme(client, row["keywords"], comments, model=MODEL)
    log.append(record)
    rows.append({"topic": topic, "status": record["status"], **(parsed.model_dump() if parsed else {})})

names = pd.DataFrame(rows).set_index("topic")
names = names.join(topics[["description", "theme"]].add_prefix("day4_"))
names["agrees_on_generic"] = names["generic_feedback"] == (names["day4_theme"] == GENERAL)
names["needs_review"] = (names["status"] != "ok") | (names["confidence"] < REVIEW_THRESHOLD) | ~names["agrees_on_generic"]
names[["theme", "confidence", "generic_feedback", "day4_theme", "needs_review"]]

,theme,confidence,generic_feedback,day4_theme,needs_review
topic,,,,,
0,Teaching delivery and interaction,0.98,False,Teaching Quality,False
1,Course content quality,0.98,False,Course Content,False
2,Exam assessment process,0.98,False,Assessment and Marking,False
3,Library resources,0.98,False,Library and Learning Resources,False
4,Laboratory practical learning,0.96,False,Practical Work,False
5,General feedback,1.00,True,General feedback (no specific theme),False
6,Extracurricular activities,0.99,False,Extracurricular Activities,False
7,General feedback,1.00,True,General feedback (no specific theme),False
8,General feedback,0.99,True,General feedback (no specific theme),False


**Result:** The LLM agrees with Day 4 on 14 of 16 topics about which are general feedback. It names all 6 real themes and 8 of the 10 general topics the same way you did. 2 topics go to review:
- topic 13 (Day 4: praise for the university as a whole) is named "Practical learning quality", confidence 0.97
- topic 14 ('Average') is named "General feedback" but has `generic_feedback=False`, confidence 0.58

**How to read it:** Find the rows with `needs_review` = True, then look for the reason: low `confidence`, or `generic_feedback` disagreeing with `day4_theme`. Topic 13 is a real disagreement. Only 2 of its 5 example comments mention practical work, so the LLM named the topic after a minority of the comments. Topic 14 contradicts itself (a general name, but flagged as not general), and its low confidence caught that. The confidence is 0.96–1.00 everywhere else, but high confidence doesn't prove a name is right; agreement with your Day 4 labels is the evidence.

Compare the LLM's descriptions with yours from Day 4. Which are better, which are worse, and why?

In [5]:
names[["theme", "description", "day4_description"]]

,theme,description,day4_description
topic,,,
0,Teaching delivery and interaction,Comments assess the quality of lecture deliver...,"Lecture delivery, interaction with students an..."
1,Course content quality,"Comments assess the relevance, quality, depth,...",Depth of courses and quality of course material
2,Exam assessment process,"Comments assess examination patterns, marks di...","Exam pattern, marks distribution and paper che..."
3,Library resources,Comments assess the library facilities and mai...,Library facilities and availability of books
4,Laboratory practical learning,Comments describe laboratory and practical wor...,Labs and practical work
5,General feedback,The comments express broadly positive opinions...,"Generic praise: good, best, great"
6,Extracurricular activities,Comments praise the availability of extracurri...,Extracurricular activities and events
7,General feedback,The comments express broadly positive opinions...,"One-word praise: excellent, nice, perfect"
8,General feedback,The comments express broadly negative opinions...,"Generic criticism: poor, bad, needs improvement"


**Result:** For the 6 real themes, the LLM's descriptions name the same subjects as yours, as full sentences. For the 10 general topics they are almost identical ("broadly positive opinions without a specific subject"), while your Day 4 notes keep the differences ('Good' vs 'Everything is fine' vs 'Average').

**How to read it:** Compare each row: does the LLM's description name the same subject as yours? If it does, the LLM could replace this manual step. If it's vaguer, information was lost. Here the LLM can do the naming for the real themes. For general feedback it collapses everything into one label, which is acceptable because those topics aren't actionable.

## 3. Task B: multi-label theme × sentiment classification

### 3a. Which comments?

Not all 724. The LLM is used where BERTopic's one-topic-per-comment can't cope: **100 comments flagged as possible multi-idea on Day 1**, plus **50 others** as a contrast group (mostly short comments, which should come back as general feedback). Fixed seed, so the same 150 comments can be gold-labelled on Day 6 and the LLM scored against them.

In [6]:
multi_idea = df[df["possible_multi_idea"]].sample(100, random_state=42)
other = df[~df["possible_multi_idea"]].sample(50, random_state=42)
sample = pd.concat([multi_idea, other])

### 3b. Classify

For each comment the LLM returns a list of labels, each with a theme, a sentiment for that theme, the **evidence** (words copied from the comment) and a confidence, plus `unlisted_topic` if the comment talks about something the taxonomy doesn't cover (a candidate **new theme**).

`classify_comment` also checks that every piece of evidence really appears in the comment (`ungrounded_evidence`).

The 150 calls run 8 at a time (`ThreadPoolExecutor`), so they take a few minutes instead of half an hour.

In [7]:
def classify(comment_id):
    parsed, record = classify_comment(client, df.loc[comment_id, "text"], taxonomy, model=MODEL)
    record["comment_id"] = comment_id
    return parsed, record


with ThreadPoolExecutor(max_workers=8) as pool:
    results = list(pool.map(classify, sample.index))

label_rows = []
for parsed, record in results:
    log.append(record)
    for label in parsed.labels if parsed else []:
        label_rows.append({"comment_id": record["comment_id"], **label.model_dump()})

labels = pd.DataFrame(label_rows)
labels["text"] = labels["comment_id"].map(df["text"])
labels["bertopic_theme"] = labels["comment_id"].map(assignments["theme"])
labels.head()

,comment_id,theme,sentiment,evidence,confidence,text,bertopic_theme
0,516,Teaching Quality,negative,teachers do not consider the new ones,0.95,"good it is, but teachers do not consider the n...",Teaching Quality
1,64,Teaching Quality,positive,teaching is good,0.99,teaching is good but some teachers are not abl...,Teaching Quality
2,64,Teaching Quality,negative,some teachers are not able to interact properl...,0.99,teaching is good but some teachers are not abl...,Teaching Quality
3,64,Teaching Quality,positive,teachers are punctual,0.99,teaching is good but some teachers are not abl...,Teaching Quality
4,576,Library and Learning Resources,negative,all the books are not avilable . If they are i...,0.99,Well... all the books are not avilable . If th...,Library and Learning Resources


One row per comment, built from the call records and the labels:

In [8]:
per_comment = pd.DataFrame([record for _, record in results]).set_index("comment_id")
per_comment = per_comment[["status", "unlisted_topic", "ungrounded_evidence"]].join(sample[["text", "possible_multi_idea"]])

by_comment = labels.groupby("comment_id")
per_comment["llm_themes"] = by_comment["theme"].agg(", ".join)
per_comment["n_themes"] = by_comment["theme"].nunique()
per_comment["n_themes"] = per_comment["n_themes"].fillna(0)
per_comment["mixed_sentiment"] = by_comment["sentiment"].nunique() > 1
per_comment["min_confidence"] = by_comment["confidence"].min()
per_comment["bertopic_theme"] = assignments["theme"]
per_comment["bertopic_agrees"] = (labels["theme"] == labels["bertopic_theme"]).groupby(labels["comment_id"]).any()

per_comment["needs_review"] = (
    (per_comment["status"] != "ok")
    | (per_comment["n_themes"] == 0)
    | (per_comment["min_confidence"] < REVIEW_THRESHOLD)
    | per_comment["unlisted_topic"].notna()
    | (per_comment["ungrounded_evidence"] != "")
)
per_comment["status"].value_counts()

status
ok    150
Name: count, dtype: int64

**Result:** All 150 calls returned `ok`: no failed or invalid responses. Every comment got at least one label, 219 labels in all.

**How to read it:** Any other status (`invalid_output`, `codex_error`, `timeout`) would mean a comment with no labels, and it would go to review. 150 of 150 means the schema and validation held for this batch. It says nothing yet about whether the labels are right; that's Day 6's gold set.

### 3c. Does the LLM add value over BERTopic?

Share of comments, split by Day 1's multi-idea flag:
- **multi_theme**: the LLM found more than one theme. BERTopic can never do this
- **mixed_sentiment**: different sentiments for different themes. The dataset's single `sentiment` label can't represent these
- **bertopic_agrees**: BERTopic's theme is among the LLM's themes. Low agreement means at least one method is wrong; section 3d shows the cases

In [9]:
per_comment["multi_theme"] = per_comment["n_themes"] >= 2
columns = ["multi_theme", "mixed_sentiment", "bertopic_agrees", "needs_review"]
per_comment.groupby("possible_multi_idea")[columns].mean().astype(float).round(2)

,multi_theme,mixed_sentiment,bertopic_agrees,needs_review
possible_multi_idea,,,,
False,0.08,0.06,0.78,0.14
True,0.22,0.42,0.80,0.08


**Result:**

| | Multi-idea (100) | Other (50) |
|---|---|---|
| 2+ themes | 22% | 8% |
| Mixed sentiment | 42% | 6% |
| BERTopic agrees | 80% | 78% |
| Needs review | 8% | 14% |

**How to read it:** Compare the two rows. Comments with Day 1's multi-idea flag get several themes or mixed sentiment 3–7 times as often, so the LLM is finding something real there, and BERTopic can't. Three caveats:
- Mixed sentiment is mostly *within* one theme. 32 of the 45 mixed comments have a single theme with two sentiments (e.g. "teaching is good but some teachers are not able to interact" gets Teaching Quality positive and negative). The dataset's single label can't show that either.
- 15 of the 31 disagreements are BERTopic outliers, which have no theme to agree with. Without them, BERTopic agrees in 119 of 135 comments (88%).
- The contrast group needs review more often because it has short, vague comments ("helps in improving skills").

Theme × sentiment across the sample: this is what the single-label dataset couldn't show, e.g. how many comments criticise marking even when they praise something else.

In [10]:
pd.crosstab(labels["theme"], labels["sentiment"], margins=True)

sentiment,negative,neutral,positive,All
theme,,,,
Assessment and Marking,13,2,13,28
Course Content,16,0,14,30
Extracurricular Activities,7,0,15,22
General feedback (no specific theme),8,1,32,41
Library and Learning Resources,13,0,11,24
Practical Work,7,0,10,17
Teaching Quality,24,4,29,57
All,88,7,124,219


**Result:** 219 labels: 124 positive, 88 negative, 7 neutral. General feedback is mostly positive (32 of 41). The actionable themes are close to half negative: Course Content 16 of 30, Teaching Quality 24 of 57, Library 13 of 24, Assessment 13 of 28.

**How to read it:** Read across a row: the negative count shows where students want change. Two cautions:
- These are labels, not comments. A comment can carry the same theme twice (38 comments do), so a theme's total overstates its number of comments.
- Two thirds of this sample are multi-idea comments, so these are not dataset-wide shares. Use Day 4's assignments for those.

### 3d. Read the output

Comments with several themes, and comments where BERTopic and the LLM disagree. For each, decide which method is right.

In [11]:
multi_theme_ids = per_comment.query("multi_theme").index[:5]
labels[labels["comment_id"].isin(multi_theme_ids)][["comment_id", "text", "theme", "sentiment", "evidence"]]

,comment_id,text,theme,sentiment,evidence
5,338,yeah quite well but ... some of the books are ...,General feedback (no specific theme),positive,quite well
6,338,yeah quite well but ... some of the books are ...,Library and Learning Resources,negative,some of the books are not available
11,60,Knowledge is maximum gained by reading books ...,Library and Learning Resources,positive,reading books
12,60,Knowledge is maximum gained by reading books ...,Teaching Quality,positive,teachers explain the concept in depth
13,386,Evaluation method is gud.But the content is mo...,Assessment and Marking,positive,Evaluation method is gud
14,386,Evaluation method is gud.But the content is mo...,Course Content,negative,the content is more to write in file
16,683,knowledge is maximum gained by reading books a...,Course Content,positive,explain depth
17,683,knowledge is maximum gained by reading books a...,Teaching Quality,positive,the teachers explain depth if we are intereste...
28,705,bcoz of lectures..we don't get time to partici...,Teaching Quality,negative,lectures
29,705,bcoz of lectures..we don't get time to partici...,Extracurricular Activities,negative,we don't get time to participate


**Result:** In these 5 comments, 386 is the clearest correct split ("Evaluation method is gud. But the content is more to write in file": Assessment positive, Course Content negative). Others are weaker:
- 338 labels "quite well" as General feedback alongside its real Library point
- 705 labels the word "lectures" as negative Teaching Quality, but the complaint is about having no time for activities
- 60 and 683 are almost the same comment but get different themes (Library + Teaching vs Course Content + Teaching)

**How to read it:** For each row, check that the `evidence` supports both the theme and the sentiment. Evidence that is one vague word ("lectures", "reading books") means the LLM split the comment further than it says. Near-duplicate comments with different labels show the labels are not fully consistent.

In [12]:
per_comment.query("status == 'ok' and bertopic_agrees == False")[["text", "bertopic_theme", "llm_themes"]].head(10)

,text,bertopic_theme,llm_themes
comment_id,,,
240,good but not good . sir improve the subject,General feedback (no specific theme),Course Content
60,Knowledge is maximum gained by reading books ...,Course Content,"Library and Learning Resources, Teaching Quality"
41,Everything is fine. Every thing is taught in d...,Unassigned (outlier),Course Content
630,Content of course is perfectly in line with th...,Unassigned (outlier),Course Content
243,Thats good but the problm is whenever in unive...,General feedback (no specific theme),Extracurricular Activities
89,The university consists of a whole world in it...,General feedback (no specific theme),Extracurricular Activities
648,Not that much good \nStudents are working so h...,Teaching Quality,General feedback (no specific theme)
423,i think mid trem should be also subjective,Course Content,Assessment and Marking
336,they are good but they should give exposure to...,Teaching Quality,"Extracurricular Activities, Extracurricular Ac..."


**Result:** 31 of 150 comments disagree. 15 of them are BERTopic outliers, so 16 are real disagreements, most often on Course Content (6 of 21 comments). In this list the LLM is usually right:
- 423 "mid term should be also subjective" is about exams (LLM: Assessment; BERTopic: Course Content)
- 243 complains that classes stop students attending university functions (LLM: Extracurricular; BERTopic: general)
- 648 ("Students are working so hard but aren't getting satisfactory result") fits neither theme, and the LLM flagged it as an unlisted topic

**How to read it:** For each row, read the text and decide which theme a person would pick. If the LLM wins most rows, BERTopic's single topic is the weaker label for these comments; if BERTopic wins, the LLM is over-reading. Outliers ("Unassigned") are a clear gain: the LLM gives them a theme (41 and 630 get Course Content).

### 3e. Analyst review queue

Everything the LLM wasn't sure about, or that failed a check. In production these go to a person who accepts, corrects or rejects them (Day 7). `unlisted_topic` values are candidate new themes for taxonomy v2.

In [13]:
review_columns = ["text", "llm_themes", "min_confidence", "unlisted_topic", "ungrounded_evidence", "status"]
review_queue = per_comment.query("needs_review")[review_columns]
review_queue

,text,llm_themes,min_confidence,unlisted_topic,ungrounded_evidence,status
comment_id,,,,,,
226,university management is extordinary but some ...,Teaching Quality,0.99,university management,,ok
76,courses are also good .but lt of subjects in c...,"Course Content, Course Content",0.96,NaN,Course Content,ok
648,Not that much good \nStudents are working so h...,General feedback (no specific theme),0.98,Academic results/outcomes,,ok
336,they are good but they should give exposure to...,"Extracurricular Activities, Extracurricular Ac...",0.56,NaN,,ok
201,Pattern is good ...marks distribution that inc...,"Assessment and Marking, Assessment and Marking",0.93,attendance,,ok
50,I WAS A STUDENT OF A BOARDING SCHOOL .....(jnv...,"Teaching Quality, Extracurricular Activities, ...",0.84,Security and student conduct/security quality,Teaching Quality,ok
22,It's good \nAlso depends on teacher's knowledg...,"General feedback (no specific theme), Teaching...",0.95,student interest,,ok
664,"Not good, becouse in lab teachers donot teach ...","Practical Work, Teaching Quality",0.90,NaN,Practical Work,ok
199,helps in improving skills,Practical Work,0.55,NaN,,ok


**Result:** 15 of 150 comments (10%) go to review: 10 for an `unlisted_topic`, 4 for ungrounded evidence and 2 for low confidence (336 at 0.56, 199 at 0.55). Comment 50 has two reasons. None are failed calls.

**How to read it:** Look at each row's reason columns.
- Unlisted topics are candidates for taxonomy v2. Class size (710), university management (226), exam preparation time (319) and atmosphere (689) are real new subjects; "a great platform" (716) and "student interest" (22) are not.
- The 4 ungrounded cases are edited quotes, not invented ones: the model fixed a typo ("lt of" became "lot of", 76), dropped words (96) or joined two fragments with "..." (50, 664). The check is strict on purpose, so these are false alarms, but they show it works.
- What the queue can't catch: a confident wrong label, like 705's "lectures" in 3d.

## 4. Task C: grounded theme summaries

One summary per actionable theme, from **all** comments Day 4 assigned to it (Day 4's assignments cover every comment; Task B only a sample). Each key point cites the comment ids behind it, and `summarise_theme` checks that every cited id was actually in the input (`unknown_ids`) and that no point is uncited (`uncited_points`).

Counts are computed here, not by the LLM: the prompt tells it not to give numbers, because LLMs count unreliably.

In [14]:
summary_rows = []
for theme in taxonomy["themes"]:
    if not theme["actionable"]:
        continue
    comments = assignments.loc[assignments["theme"] == theme["name"], "text"].to_dict()
    parsed, record = summarise_theme(client, theme["name"], theme["description"], comments, model=MODEL)
    log.append(record)
    for point in parsed.key_points if parsed else []:
        summary_rows.append({"theme": theme["name"], "summary": parsed.summary, **point.model_dump()})

summaries = pd.DataFrame(summary_rows)
summaries.drop_duplicates("theme")[["theme", "summary"]]

,theme,summary
0,Teaching Quality,"Students generally describe teaching, punctual..."
10,Course Content,Students generally describe the course content...
16,Assessment and Marking,Students generally praise the examination patt...
28,Library and Learning Resources,Students generally praise the library faciliti...
35,Practical Work,Students generally find practical work and lab...
42,Extracurricular Activities,Students generally praise the university for o...


**Result:** 6 summaries, one per actionable theme, from all 399 comments Day 4 assigned to them (36–87 per theme). All 6 open with "Students generally praise/describe ... as good", then list the complaints.

**How to read it:** The opening sentence is the claim to check: is the theme really mostly positive? The crosstab in 3c has about half of the actionable labels negative. "Generally" is a frequency claim, which the prompt told the LLM not to make, so treat it as unverified until you count it in pandas.

The key points, one row each. To audit a point, look up its ids in `df`: does every cited comment really say that?

In [15]:
summaries[["theme", "point", "sentiment", "supporting_comment_ids"]]

,theme,point,sentiment,supporting_comment_ids
0,Teaching Quality,Many students praise the overall quality of te...,positive,"[20, 24, 57, 62, 66, 82, 107, 134, 153, 171, 1..."
1,Teaching Quality,Teaching quality and lecture delivery vary con...,neutral,"[3, 22, 23, 159, 237, 238, 244, 280, 297, 360,..."
2,Teaching Quality,"Some students find lecture delivery unclear, i...",negative,"[7, 23, 133, 151, 185, 244, 270, 360, 391, 430..."
3,Teaching Quality,Interaction with students is viewed positively...,neutral,"[20, 57, 88, 106, 164, 170, 171, 172, 220, 260..."
4,Teaching Quality,Students say some teachers do not engage all s...,negative,"[31, 88, 164, 336, 516]"
5,Teaching Quality,Some students report concerns about teacher ex...,negative,"[101, 159, 226, 244, 297, 365, 387, 527, 633]"
6,Teaching Quality,Punctuality is widely described as good and co...,positive,"[7, 20, 57, 62, 66, 107, 172, 189, 325, 330, 3..."
7,Teaching Quality,Students mention perceived partiality or unfai...,negative,"[299, 426, 527]"
8,Teaching Quality,"Students would like more practical knowledge, ...",negative,"[304, 365, 336, 546, 613, 677]"
9,Teaching Quality,Some students feel the teaching schedule or wo...,negative,"[69, 125, 662, 705]"


**Result:** 49 key points (6–12 per theme): 30 negative, 15 positive, 4 neutral. Every point cites 1–29 comments, and the id checks found no unknown ids and no uncited points (section 8's report).

**How to read it:** Check two things on each row.
- Does the `sentiment` match the `point`? Points 13 ("praised by some students but criticized by others") and 22 ("Students disagree about negative marking") describe mixed views but are tagged negative; neutral would fit better.
- How many ids back the point? A point with one id (27, "One student praises the seating arrangement"; 47, politics in cricket) is a single opinion. It shouldn't carry the same weight as a point with 20+ ids.

The id checks prove the ids are real, not that the comments say what the point claims. The next cell checks that.

In [16]:
df.loc[summaries.loc[0, "supporting_comment_ids"], "text"]

20     Interaction is satisfying, Lecture delivery is...
24     All lectures are giving their best to teach th...
57     intraction is good and leacture delivery also ...
62     our university is good in terms of lecture del...
66          teachers are good in subject,and punctuality
82     excellent ...they  work hard..their way of tea...
107    Excellent lectures are delivered by teachers a...
134                       everything is good for faculty
153    This semester university has provide best teac...
171    overall rating on this activity is 4.5/5 .very...
172    interaction satisfied, lecture delivery good,p...
189        maintained punctuality, good lecture delivery
292    teaching activities of our university is good....
325    the staff is highly punctual ad interactive an...
330    In this university there is good interaction w...
371    very good teaching facilities, lecture given a...
474    excellent university  the mode of teaching is ...
493                            

**Result:** Point 0 ("Many students praise the overall quality of teaching, lecture delivery, punctuality, and teacher helpfulness") cites 27 comments. All 27 praise teaching, lecture delivery or punctuality (e.g. 189 "maintained punctuality, good lecture delivery"), so this point is grounded.

**How to read it:** A point is grounded if every cited comment says what it claims. Change the `0` to another row number to audit more points. Try the ones most likely to overstate: a negative point, and a point with only one or two ids.

## 5. Call log: latency, cost and failures

Every call from every task, including failed ones. With Codex, `cost_usd` shows 0: usage counts against your ChatGPT plan instead, so compare tokens.

In [17]:
calls = pd.DataFrame(log)
calls.groupby(["task", "status"]).agg(
    calls=("task", "size"),
    mean_latency_s=("latency_s", "mean"),
    input_tokens=("input_tokens", "sum"),
    output_tokens=("output_tokens", "sum"),
    cost_usd=("cost_usd", "sum"),
).round(3)

,,calls,mean_latency_s,input_tokens,output_tokens,cost_usd
task,status,,,,,
classify_theme,ok,150,12.630,1829597,20359,0.0
name_theme,ok,17,6.684,201627,943,0.0
summarise_theme,ok,6,9.778,79819,4020,0.0


**Result:** 173 calls, all `ok`. Per call, classifying takes ~12,200 input and ~140 output tokens, naming ~11,900 and ~55, summarising ~13,300 and ~670. The mean latency is 6.7–12.6 s, and the 150 classifications took about 4 minutes with 8 in parallel.

**How to read it:** Divide `input_tokens` by `calls`. About 12,000 tokens per call whatever the task means most of the input is Codex's own agent instructions, not your prompt: the same prompts through the Claude API are about 1,000 input tokens each. `cost_usd` shows 0 because NaN sums to 0 in pandas; on Codex the real cost is usage of your ChatGPT plan.

## 6. Save outputs

In [ ]:
failed_calls = calls.query("status != 'ok'")[["task", "status", "raw_output"]]
naming_review = names.query("needs_review")[["theme", "confidence", "generic_feedback", "day4_theme"]]
summary_checks = calls.query("task == 'summarise_theme'")[["theme", "status", "unknown_ids", "uncited_points"]]

report = f'''# LLM Failure Examples

Model: {MODEL}

## Failed calls
{failed_calls.to_string()}

## Theme naming: needs review
{naming_review.to_string()}

## Classification: review queue
{review_queue.to_string()}

## Summaries: id checks
{summary_checks.to_string()}

'''

(RESULTS / "llm_failure_examples.md").write_text(report)
names.to_csv(RESULTS / "llm_theme_names.csv")
labels.to_csv(RESULTS / "llm_theme_assignments.csv", index=False)
summaries.to_csv(RESULTS / "grounded_summaries.csv", index=False)
calls.drop(columns="raw_output").to_csv(RESULTS / "llm_call_log.csv", index=False)